## Script used to calculate LCL heights for analysis

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime as dt
import itertools
from math import e
from datetime import datetime, timedelta
import glob
import metpy.calc as mpcalc
from metpy.units import units
import warnings; warnings.simplefilter('ignore')
from matplotlib.dates import date2num, DateFormatter
import matplotlib.dates as mdates
import xarray as xr
from matplotlib.dates import date2num, DateFormatter
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
import matplotlib
matplotlib.rc('xtick', labelsize=13) 
matplotlib.rc('ytick', labelsize=13)
# import datetime

In [2]:
dataPath = (
    "/Users/cneumaie/OneDrive - Colostate/Research/Drone_Profiles/00_Data/"
)

figPath = "/Users/cneumaie/OneDrive - Colostate/Research/Drone_Profiles/02_Figures/"

In [3]:
def rh_from_dewpt(TD, T):
    return 100*(np.exp((17.625*TD)/(243.04+TD))/np.exp((17.625*T)/(243.04+T))) 

In [133]:
def open_iMet(imet_filepath):
    '''Opens iMet Sounding to a pandas dataframe'''
    df = pd.read_table(open(imet_filepath, errors='ignore'),   ###reading in file as a pandas dataframe
                  skiprows=[1,2], sep=r'\s+', index_col = False)
    # df['Temp'] = df['Temp']+273.15 ###Converts to K
    # df["Temp_d"] = df["DP"]+273.15
    # df["Press"] = df["press_Pa"]/100
    # df["Rel_Hum"] = df["RH_pct"]
    df["Temp_d"] = mpcalc.dewpoint_from_relative_humidity(df["Temp"] * units.K, df["Rel_Hum"] * units.percent)
    # df["Mix_Rat"] = mpcalc.mixing_ratio_from_relative_humidty(df["Press"]*units.hPa,df["Temp"] * units.K, df["Rel_Hum"] * units.percent)
    df["Theta"] = mpcalc.potential_temperature(df["Press"].values*units.hPa,
                                               df["Temp"].values*units.K)
                                               
    
    return df

In [179]:
def open_iMet_qc(imet_filepath):
    '''Opens iMet Sounding to a pandas dataframe'''
    df = pd.read_csv(imet_filepath)
    df = df.dropna()
    # df['Temp'] = df['Temp']+273.15 ###Converts to K
    # df["Temp_d"] = df["DP"]+273.15
    df["Temp"] = df["Tair_K"]
    df["Press"] = df["press_Pa"]/100
    df["Rel_Hum"] = df["RH_pct"]
    df["Temp_d"] = mpcalc.dewpoint_from_relative_humidity(df["Temp"].values * units.K, df["Rel_Hum"].values * units.percent)
    df["Mix_Rat"] = mpcalc.mixing_ratio_from_relative_humidity(df["Press"].values*units.hPa,df["Temp"].values * units.K, df["Rel_Hum"].values * units.percent).to("g/kg")
    df["Theta"] = mpcalc.potential_temperature(df["Press"].values*units.hPa,
                                               df["Temp"].values*units.K)
    return df

In [5]:
# Version 1.0 released by David Romps on September 12, 2017.
# Version 1.1 vectorized lcl.R, released on May 24, 2021.
# 
# When using this code, please cite:
# 
# @article{16lcl,
#   Title   = {Exact expression for the lifting condensation level},
#   Author  = {David M. Romps},
#   Journal = {Journal of the Atmospheric Sciences},
#   Year    = {2017},
#   Month   = dec,
#   Number  = {12},
#   Pages   = {3891--3900},
#   Volume  = {74}
# }
#
# This lcl function returns the height of the lifting condensation level
# (LCL) in meters.  The inputs are:
# - p in Pascals
# - T in Kelvins
# - Exactly one of rh, rhl, and rhs (dimensionless, from 0 to 1):
#    * The value of rh is interpreted to be the relative humidity with
#      respect to liquid water if T >= 273.15 K and with respect to ice if
#      T < 273.15 K. 
#    * The value of rhl is interpreted to be the relative humidity with
#      respect to liquid water
#    * The value of rhs is interpreted to be the relative humidity with
#      respect to ice
# - return_ldl is an optional logical flag.  If true, the lifting deposition
#   level (LDL) is returned instead of the LCL. 
# - return_min_lcl_ldl is an optional logical flag.  If true, the minimum of the
#   LCL and LDL is returned.

def lcl(p,T,rh=None,rhl=None,rhs=None,return_ldl=False,return_min_lcl_ldl=False):

   import math
   import scipy.special

   # Parameters
   Ttrip = 273.16     # K
   ptrip = 611.65     # Pa
   E0v   = 2.3740e6   # J/kg
   E0s   = 0.3337e6   # J/kg
   ggr   = 9.81       # m/s^2
   rgasa = 287.04     # J/kg/K 
   rgasv = 461        # J/kg/K 
   cva   = 719        # J/kg/K
   cvv   = 1418       # J/kg/K 
   cvl   = 4119       # J/kg/K 
   cvs   = 1861       # J/kg/K 
   cpa   = cva + rgasa
   cpv   = cvv + rgasv

   # The saturation vapor pressure over liquid water
   def pvstarl(T):
      return ptrip * (T/Ttrip)**((cpv-cvl)/rgasv) * \
         math.exp( (E0v - (cvv-cvl)*Ttrip) / rgasv * (1/Ttrip - 1/T) )
   
   # The saturation vapor pressure over solid ice
   def pvstars(T):
      return ptrip * (T/Ttrip)**((cpv-cvs)/rgasv) * \
         math.exp( (E0v + E0s - (cvv-cvs)*Ttrip) / rgasv * (1/Ttrip - 1/T) )

   # Calculate pv from rh, rhl, or rhs
   rh_counter = 0
   if rh  is not None:
      rh_counter = rh_counter + 1
   if rhl is not None:
      rh_counter = rh_counter + 1
   if rhs is not None:
      rh_counter = rh_counter + 1
   if rh_counter != 1:
      print(rh_counter)
      exit('Error in lcl: Exactly one of rh, rhl, and rhs must be specified')
   if rh is not None:
      # The variable rh is assumed to be 
      # with respect to liquid if T > Ttrip and 
      # with respect to solid if T < Ttrip
      if T > Ttrip:
         # print(pvstarl(T))
         pv = rh * pvstarl(T)
      else:
         pv = rh * pvstars(T)
      rhl = pv / pvstarl(T)
      rhs = pv / pvstars(T)
   elif rhl is not None:
      pv = rhl * pvstarl(T)
      rhs = pv / pvstars(T)
      if T > Ttrip:
         rh = rhl
      else:
         rh = rhs
   elif rhs is not None:
      pv = rhs * pvstars(T)
      rhl = pv / pvstarl(T)
      if T > Ttrip:
         rh = rhl
      else:
         rh = rhs
   if pv > p:
    # print(pv)
    return np.nan

   # Calculate lcl_liquid and lcl_solid
   qv = rgasa*pv / (rgasv*p + (rgasa-rgasv)*pv)
   rgasm = (1-qv)*rgasa + qv*rgasv
   cpm = (1-qv)*cpa + qv*cpv
   if rh == 0:
      return cpm*T/ggr
   aL = -(cpv-cvl)/rgasv + cpm/rgasm
   bL = -(E0v-(cvv-cvl)*Ttrip)/(rgasv*T)
   cL = pv/pvstarl(T)*math.exp(-(E0v-(cvv-cvl)*Ttrip)/(rgasv*T))
   aS = -(cpv-cvs)/rgasv + cpm/rgasm
   bS = -(E0v+E0s-(cvv-cvs)*Ttrip)/(rgasv*T)
   cS = pv/pvstars(T)*math.exp(-(E0v+E0s-(cvv-cvs)*Ttrip)/(rgasv*T))
   lcl = cpm*T/ggr*( 1 - \
      bL/(aL*scipy.special.lambertw(bL/aL*cL**(1/aL),-1).real) )
   ldl = cpm*T/ggr*( 1 - \
      bS/(aS*scipy.special.lambertw(bS/aS*cS**(1/aS),-1).real) )

   # Return either lcl or ldl
   if return_ldl and return_min_lcl_ldl:
      exit('return_ldl and return_min_lcl_ldl cannot both be true')
   elif return_ldl:
      return ldl
   elif return_min_lcl_ldl:
      return min(lcl,ldl)
   else:
      return lcl

In [112]:
sonde["Mix_Rat"]

0       15.2
1       14.8
2       14.6
3       14.2
4       13.6
        ... 
5458     0.0
5459     0.0
5460     0.0
5461     0.0
5462     0.0
Name: Mix_Rat, Length: 5463, dtype: float64

In [128]:
test = pd.read_csv(glob.glob(f"{path}/BACS*.csv")[0])

In [165]:
test = test.dropna()

In [166]:
test

,time_MDT,hgtAGL_m,lon_deg,lat_deg,Tair_K,RH_pct,press_Pa,wspeed_mps,wdir_deg,vvel_mps,Tair_QC,RH_QC,press_QC,wind_QC
3,2023-06-21 13:20:00,13.6,-104.77744,40.80871,297.27,61.78,83321.0,3.4,103.0,6.3,0,0,0,0
4,2023-06-21 13:20:01,20.3,-104.77749,40.80874,296.83,60.68,83257.0,3.1,104.0,6.3,0,0,0,0
5,2023-06-21 13:20:02,27.2,-104.77752,40.80877,296.65,60.88,83191.0,2.9,101.0,6.4,0,0,0,0
6,2023-06-21 13:20:03,34.4,-104.77757,40.80878,296.46,60.93,83123.0,2.9,97.0,6.4,0,0,0,0
7,2023-06-21 13:20:04,41.7,-104.77764,40.80875,296.21,61.06,83054.0,3.0,95.0,6.4,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5458,2023-06-21 14:50:55,20272.1,-104.54258,41.33295,216.65,4.00,4316.0,3.7,108.0,-0.8,0,0,0,0
5459,2023-06-21 14:50:56,20276.0,-104.54261,41.33294,216.67,4.13,4313.0,3.7,109.0,-2.0,0,0,0,0
5460,2023-06-21 14:50:57,20280.2,-104.54263,41.33293,216.68,4.14,4310.0,3.7,109.0,-3.4,0,0,0,0
5461,2023-06-21 14:50:58,20284.4,-104.54266,41.33292,216.68,4.16,4308.0,3.9,111.0,-4.7,0,0,0,0


# Get radiosonde data

In [219]:
radiosondePath = f"{dataPath}radiosonde/environment/" 
rad_folders = [x[0] for x in sorted(os.walk(radiosondePath))][1:]
lcl_list = []
for path in rad_folders:
    print(path)
    sonde = open_iMet_qc(glob.glob(f"{path}/BACS*.csv")[0])
    hgt = 500
    press = 100
    # print(sonde["Press"])
    # bottom = sonde["Alt_AGL"].min()
    # sonde_a = sonde.loc[sonde["Alt_AGL"].between(bottom, bottom + hgt)].drop(columns="Alt_AGL").mean(numeric_only=True)
    bottom_p = sonde["Press"].max()*units.hPa
    bottom_z = sonde.iloc[sonde["Press"].idxmax()]["hgtAGL_m"]    
    t_d = sonde["Temp_d"]
    # bottom_h = sonde["Alt_AGL"].max()*units.m
    # sonde_a = sonde.loc[sonde["Press"].between(bottom - press,bottom)].mean(numeric_only=True)
    theta, rv = mpcalc.mixed_layer(sonde["Press"].values*units.hPa, sonde["Theta"].values*units.K, sonde["Mix_Rat"].values*units("g/kg"),
                              bottom = bottom_p,depth = 50*units.hPa)
    p,t_test,td_test = mpcalc.mixed_parcel(sonde["Press"].values*units.hPa, sonde["Temp"].values*units.K, t_d.values*units.K,
                               bottom = bottom_p,depth = 50*units.hPa) 
    # print(td_test)
    t = theta.magnitude*((bottom_p.magnitude/1000)**(0.286))
    # t = mpcalc.temperature_from_potential_temperature(p,theta)
    rh = mpcalc.relative_humidity_from_mixing_ratio(bottom_p,t*units.K,rv).to('percent')
    # rh2 = mpcalc.relative_humidity_from_dewpoint(t_test,td_test)
    # print(p,t,rh)
    # print(sonde_a)
    # print(t)
    # sond_200_loc = sonde[sonde["Alt_AGL"] ==
    # lcl_calc = lcl(sonde["Press"][:70].mean()*100,sonde["Temp"][:70].mean(), rh = sonde["RelHum"][:70].mean()/100)
    # lcl_calc = lcl(sonde_a["Press"]*100,sonde_a["Temp"], rh = sonde_a["RelHum"]/100)
    lcl_calc = lcl(p.magnitude*100,t,rh.magnitude/100)
    # lcl_calc = lcl(sonde["Press"].max()*100,sonde.iloc[sonde["Press"].idxmax()]["Temp"],sonde.iloc[sonde["Press"].idxmax()]["Rel_Hum"]/100)
    # print(lcl_calc)
    # print(bottom_p,t, t_test,rv,rh, rh2)
    # print(bottom_p, sonde.iloc[sonde["Press"].idxmax()]["Temp"],sonde.iloc[sonde["Press"].idxmax()]["Rel_Hum"])
    lcl_calc -= bottom_z
    lcl_list.append(lcl_calc)
# radiosonde_23 = open_iMet(glob.glob(f"{radiosondePath}20230610_BACS0{r}/*TSPOTINT.txt")[0])
# ### IOP 26 CP1
# r = 3
# radiosonde_26 = open_iMet(glob.glob(f"{radiosondePath}20230617_BACS0{r}/*TSPOTINT.txt")[0])

/Users/cneumaie/OneDrive - Colostate/Research/Drone_Profiles/00_Data/radiosonde/environment/20230527_BACS01
/Users/cneumaie/OneDrive - Colostate/Research/Drone_Profiles/00_Data/radiosonde/environment/20230530_BACS01
/Users/cneumaie/OneDrive - Colostate/Research/Drone_Profiles/00_Data/radiosonde/environment/20230601_BACS01
/Users/cneumaie/OneDrive - Colostate/Research/Drone_Profiles/00_Data/radiosonde/environment/20230602_BACS01
/Users/cneumaie/OneDrive - Colostate/Research/Drone_Profiles/00_Data/radiosonde/environment/20230608_BACS01
/Users/cneumaie/OneDrive - Colostate/Research/Drone_Profiles/00_Data/radiosonde/environment/20230610_BACS01
/Users/cneumaie/OneDrive - Colostate/Research/Drone_Profiles/00_Data/radiosonde/environment/20230612_BACS01
/Users/cneumaie/OneDrive - Colostate/Research/Drone_Profiles/00_Data/radiosonde/environment/20230615_BACS01
/Users/cneumaie/OneDrive - Colostate/Research/Drone_Profiles/00_Data/radiosonde/environment/20230617_BACS01
/Users/cneumaie/OneDrive - C

In [220]:
lcl_list

[911.598500617625,
 1862.6776732850985,
 1226.0914838874683,
 1218.7731052897602,
 1797.4630200769145,
 1648.351281469615,
 836.5259340479708,
 1861.409689483476,
 917.3851479712785,
 963.452228539639]

In [87]:
# lcl_list_500m = np.copy(lcl_list)

In [85]:
# lcl_list_100m = np.copy(lcl_list)

In [118]:
# lcl_list_100p = np.copy(lcl_list)

In [122]:
# lcl_list_50p = np.copy(lcl_list)